In [33]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [3]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

In [7]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Q1

In [8]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

The most frequent observation for the industry column is 'technology'

## Q2

In [9]:
df[['interaction_count','lead_score','number_of_courses_viewed','annual_income']].corr()

,interaction_count,lead_score,number_of_courses_viewed,annual_income
interaction_count,1.000000,0.915746,0.721609,0.122842
lead_score,0.915746,1.000000,0.757204,0.229496
number_of_courses_viewed,0.721609,0.757204,1.000000,0.161300
annual_income,0.122842,0.229496,0.161300,1.000000


Option 1, interaction_count and lead_score, has the highest coefficient of 0.916(3 dp)

## Q3

In [10]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [15]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

categorical_mi = df_full_train[categorical].apply(mutual_info_converted_score)
round(categorical_mi, 2)

lead_source          0.03
industry             0.00
employment_status    0.02
location             0.00
dtype: float64

Lead score has the biggest mutual information score of 0.03

## Q4

In [27]:
dv = DictVectorizer(sparse=False)
train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)
X_train.shape

(3000, 28)

In [24]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)
X_val.shape

(1000, 28)

In [28]:
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [42]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:,1]
convert_decision = (y_pred >= 0.5)
acc = (y_val == convert_decision).mean()
print(acc)

0.645


The accuracy of the model on the validation dataset is 0.65(2dp)

## Q5

In [51]:
acc_score = []
removed_features = ['lead_source','number_of_courses_viewed','interaction_count']

for removed in removed_features:
    categorical = ['lead_source','industry','employment_status','location']
    numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
    if removed in categorical:
        categorical.remove(removed)
    else:
        numerical.remove(removed)
    df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    df_train, df_val = train_test_split(
        df_full_train, test_size=0.25, random_state=42
    )
    y_train = df_train.converted.values
    y_val = df_val.converted.values
    y_test = df_test.converted.values
    del df_train['converted']
    del df_val['converted']
    del df_test['converted']

    dv = DictVectorizer(sparse=False)
    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:,1]
    convert_decision = (y_pred >= 0.5)
    adj_acc = (y_val == convert_decision).mean()
    acc_score.append(abs(adj_acc - acc))

index = acc_score.index(min(acc_score))
print(f'The feature with the smallest difference is {removed_features[index]} with a difference of {min(acc_score)}')

The feature with the smallest difference is number_of_courses_viewed with a difference of 0.0020000000000000018


## Q6

In [55]:
C = [0.000001, 0.00001, 0.0001, 0.001]
acc_score = []
for parameter in C:
    categorical = ['lead_source','industry','employment_status','location']
    numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
    df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    df_train, df_val = train_test_split(
        df_full_train, test_size=0.25, random_state=42
    )
    y_train = df_train.converted.values
    y_val = df_val.converted.values
    y_test = df_test.converted.values
    del df_train['converted']
    del df_val['converted']
    del df_test['converted']

    dv = DictVectorizer(sparse=False)
    train_dict = df_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[categorical + numerical].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=parameter, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:,1]
    convert_decision = (y_pred >= 0.5)
    adj_acc = (y_val == convert_decision).mean()
    acc_score.append(abs(adj_acc - acc))
index = acc_score.index(max(acc_score))
print(f'The C value with the best accuracy is {C[index]:10f} with an accuracy of {round(acc_score[index],3)}')

The C value with the best accuracy is   0.000001 with an accuracy of 0.047
